In [7]:
import os, json, cv2, numpy as np, matplotlib.pyplot as plt
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import masks_to_boxes

import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F

import albumentations as A # Library for augmentations

In [8]:
import transforms, utils, engine, train
from utils import collate_fn
from engine import train_one_epoch, evaluate
import cv2

from model_utilities import parse_annotation, precheck_annotation

In [9]:
def train_transform():
    return A.Compose([
        A.Sequential([
            A.RandomRotate90(p=1), # Random rotation of an image by 90 degrees zero or more times
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, brightness_by_max=True, always_apply=False, p=1), # Random change of brightness & contrast
        ], p=1)
    ],
    keypoint_params=A.KeypointParams(format='xy'), # More about keypoint formats used in albumentations library read at https://albumentations.ai/docs/getting_started/keypoints_augmentation/
)

In [10]:
def img_json_pairs(dir):
    """
    match all vein image and annotation file pairs
    """
    files = os.listdir(dir)
    veinans = list(filter(lambda x : ("vein" in x.lower()) and x.endswith(".json"), files))
    veinims = list(filter(lambda x : ("vein" in x.lower()) and x.lower().endswith(".jpg"), files))
    pairs = []
    for im in veinims:
        imname = im.lower().replace(".jpg","")
        for an in veinans:
            if imname == an.lower().replace("_labels","").replace("_label","").replace(".json",""):
                pairs.append((dir+"/"+im,dir+"/"+an))
    return pairs

def grab_all_data(dir):
    """ 
    get all vien images from the subdirectories of a directory
    """
    all_pairs = []
    for sub in os.listdir(dir):
        if os.path.isdir(dir + "/" + sub):
            subdir_pairs = img_json_pairs(dir + "/"+ sub)
            # temp fix: try to parse the annotation before adding it, if it doesnt work, dump it
            for pair in subdir_pairs:
                try:
                    precheck_annotation(pair[1])
                except Exception:
                    subdir_pairs.remove(pair)
                    print(Exception)
            all_pairs = all_pairs + subdir_pairs
    return all_pairs


def shuffle_split_train_test(data, ratio):
    """
    split the data into training and testing sets, sizes defined by the ratio of training to testing
    returns (test, train)
    """
    random.shuffle(data)
    test = data[:int(len(data) * ratio)]
    train = data[int(len(data) * ratio):]
    return(test, train)
# split_train_test([1,2,3,4,5,6,8,9,10],0.2)

    

In [11]:
def keypoints_to_bbox(keypoints, offset):
    x_coordinates = []
    y_coordinates = []
    for kp in keypoints[0]:
        x_coordinates.append(kp[0])
        y_coordinates.append(kp[1])

    xmin = min(x_coordinates)
    ymin = min(y_coordinates)
    xmax = max(x_coordinates)
    ymax = max(y_coordinates)

    # print([xmin-offset, ymin-offset, xmax+offset, ymax+offset])

    bbox = np.array([[xmin-offset, ymin-offset, xmax+offset, ymax+offset]])

    return bbox

In [12]:
# alternate bounding box using the 

# takes return from cv2.imread("path", 0)

def get_vein_shape(im):

    # im = img_util.remove_edge(im) # crop out any edges that might mess up shape recognition

    blur = cv2.GaussianBlur(im, (5, 5), 0)
    th, im_th = cv2.threshold(blur, 10, 255, cv2.THRESH_BINARY) # seperate light from dark look at threshold image later
    
    label_count, labels, stats, centroids = cv2.connectedComponentsWithStats(im_th, 4, cv2.CV_32S)
    sizes = stats[:,-1]

    max_label = 1
    
    max_size = sizes[1]
    for i in range(2, label_count):
        if sizes[i] > max_size:
            max_label = i
            max_size = sizes[i]
    vein_label = np.array([max_label])
    petal = cv2.inRange(labels, vein_label, vein_label)
    return petal

In [13]:
# test alternate bounding boxes

PATH = "/Users/oliverbaltzer/Google Drive/Shared drives/Cooley_Lab/Hybrid Speckling_Spot-Vein-Project/LeahSamuels_PetalPhotos/p117/F1P117_Vein_Side2_210804.jpg"

# visualize_kp(get_vein_shape(cv2.imread(PATH, 0)), [],[])

In [14]:
class ClassDataset(Dataset):
    def __init__(self, data, transform=None, demo=False):                
        self.transform = transform
        self.demo = demo # Use demo=True if you need transformed and original images (for example, for visualization purposes)
        self.impairs = data
    
    def __getitem__(self, idx):
        impair = self.impairs[idx]
        img_path = impair[0]
        annotations_path = impair[1]
        # print(annotations_path)
        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)
        keypoints_original = parse_annotation(annotations_path)
        bboxes_original = np.array(masks_to_boxes(torch.as_tensor([get_vein_shape(cv2.imread(PATH, 0))])))
        bboxes_labels_original = ["leaf" for _ in bboxes_original]


        # print(f"keypoints: {keypoints_original}")
        # print(f"original kp shape: {np.shape(keypoints_original)}")
        # print(f"bounding boxes: {bboxes_original}")
        # print(f"bounding box labels: {bboxes_labels_original}")


        if self.transform:   
            # Converting keypoints from [x,y,visibility]-format to [x, y]-format + Flattening nested list of keypoints            
            # For example, if we have the following list of keypoints for three objects (each object has two keypoints):
            # [[obj1_kp1, obj1_kp2], [obj2_kp1, obj2_kp2], [obj3_kp1, obj3_kp2]], where each keypoint is in [x, y]-format            
            # Then we need to convert it to the following list:
            # [obj1_kp1, obj1_kp2, obj2_kp1, obj2_kp2, obj3_kp1, obj3_kp2]
            # print(f"original keypoints{keypoints_original}")
            keypoints_original_flattened = [el[0:2] for kp in keypoints_original for el in kp]
            # print(f"original flattened keypoints{keypoints_original_flattened}")
            
            # Apply augmentations
            transformed = self.transform(image=img_original, bboxes=bboxes_original, bboxes_labels=bboxes_labels_original, keypoints=keypoints_original_flattened)
            img = transformed['image']
            bboxes = transformed['bboxes']
            
            # Unflattening list transformed['keypoints']
            # For example, if we have the following list of keypoints for three objects (each object has two keypoints):
            # [obj1_kp1, obj1_kp2, obj2_kp1, obj2_kp2, obj3_kp1, obj3_kp2], where each keypoint is in [x, y]-format
            # Then we need to convert it to the following list:
            # [[obj1_kp1, obj1_kp2], [obj2_kp1, obj2_kp2], [obj3_kp1, obj3_kp2]]
            keypoints_transformed_unflattened = np.reshape(np.array(transformed['keypoints']), (1,4,2)).tolist()
            # print(f"transformed unflattened: {keypoints_transformed_unflattened}")
            # Converting transformed keypoints from [x, y]-format to [x,y,visibility]-format by appending original visibilities to transformed coordinates of keypoints
            keypoints = []
            # print(f"original keypoints: {keypoints_original}")
            for o_idx, obj in enumerate(keypoints_transformed_unflattened): # Iterating over objects
                obj_keypoints = []
                # print(obj)
                # print(keypoints_original[o_idx])
                for k_idx, kp in enumerate(obj): # Iterating over keypoints in each object
                    # kp - coordinates of keypoint
                    # keypoints_original[o_idx][k_idx][2] - original visibility of keypoint
                    obj_keypoints.append(kp + [keypoints_original[o_idx][k_idx][2]])
                keypoints.append(obj_keypoints)
        
        else:
            img, bboxes, keypoints = img_original, bboxes_original, keypoints_original        
        
        # Convert everything into a torch tensor        
        bboxes = torch.as_tensor(bboxes, dtype=torch.float32)       
        target = {}
        target["boxes"] = bboxes
        target["labels"] = torch.as_tensor([1 for _ in bboxes], dtype=torch.int64)
        target["image_id"] = int(idx)
        target["area"] = (bboxes[:, 3] - bboxes[:, 1]) * (bboxes[:, 2] - bboxes[:, 0])
        target["iscrowd"] = torch.zeros(len(bboxes), dtype=torch.int64)
        target["keypoints"] = torch.as_tensor(keypoints, dtype=torch.float32)        
        img = F.to_tensor(img)
        
        bboxes_original = torch.as_tensor(bboxes_original, dtype=torch.float32)
        target_original = {}
        target_original["boxes"] = bboxes_original
        target_original["labels"] = torch.as_tensor([1 for _ in bboxes_original], dtype=torch.int64)
        target_original["image_id"] = int(idx)
        target_original["area"] = (bboxes_original[:, 3] - bboxes_original[:, 1]) * (bboxes_original[:, 2] - bboxes_original[:, 0])
        target_original["iscrowd"] = torch.zeros(len(bboxes_original), dtype=torch.int64)
        target_original["keypoints"] = torch.as_tensor(keypoints_original, dtype=torch.float32)        
        img_original = F.to_tensor(img_original)

        if self.demo:
            return img, target, img_original, target_original
        else:
            return img, target
    
    def __len__(self):
        return len(self.impairs)

In [15]:
import torch
import random
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
from PIL import ImageDraw, Image


def visualize_kp(image, bbox, keypoints):
    """
    visualize a specific image and keypoints
    """
    if isinstance(image, torch.Tensor):
        image = F.to_pil_image(image)
    elif isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    draw = ImageDraw.Draw(image)


    for obj_kpts in keypoints:
        for kp in obj_kpts:
            x, y = float(kp[0]), float(kp[1])
            draw.ellipse(
                [(x - 4, y - 4),
                    (x + 4, y + 4)],
                outline="red", width=2)
    # Visualize bounding box

    for box in bbox:
        x_min, y_min, x_max, y_max = box
        draw.rectangle(
            [int(x_min),int(y_min),int(x_max),int(y_max)],
            outline="blue", width=2
        )


    # Show result
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted keypoints")
    plt.show()


def visualize_random_keypoints(dataset, keypoint_radius=4):
    """
    Visualizes keypoints on a random image from a PyTorch dataset.
    
    Assumes dataset returns (image, target) where:
      - image: Tensor [3, H, W] or PIL Image
      - target: dict with 'keypoints' of shape [N, K, 3]
    """
    idx = random.randint(0, len(dataset) - 1)
    # print(len(dataset[idx]))
    # print(dataset[idx])
    _ , _ ,image, target = dataset[idx]
    # Convert tensor image to PIL
    if isinstance(image, torch.Tensor):
        image = F.to_pil_image(image)

    draw = ImageDraw.Draw(image)

    # Supports multiple objects per image (N)
    keypoints_batch = target["keypoints"].tolist()  # shape [N, K, 3]
    for obj_kpts in keypoints_batch:
        for kp in obj_kpts:
            if kp[2] > 0:  
                x, y = float(kp[0]), float(kp[1])
                draw.ellipse(
                    [(x - keypoint_radius, y - keypoint_radius),
                        (x + keypoint_radius, y + keypoint_radius)],
                    outline="red", width=2
                )
    # Visualize bounding box

    if "boxes" in target:
        bbox = target["boxes"]
        x_min, y_min, x_max, y_max = bbox.tolist()[0]
        draw.rectangle(
            [int(x_min),int(y_min),int(x_max),int(y_max)],
            outline="blue", width=2
        )


    # Show result
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Sample #{idx} — Keypoints visualized")
    plt.show()



In [16]:

KEYPOINTS_FOLDER_TRAIN = '/Users/oliverbaltzer/Google Drive/Shared drives/Cooley_Lab/Hybrid Speckling_Spot-Vein-Project/JoshuaShin_PetalPhotos/'
dataset = ClassDataset(grab_all_data(KEYPOINTS_FOLDER_TRAIN), transform=train_transform(), demo=True)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
visualize_random_keypoints(dataset)

iterator = iter(data_loader)
batch = next(iterator)

print("Original targets:\n", batch[3], "\n\n")
print("Transformed targets:\n", batch[1])


<class 'Exception'>


KeyboardInterrupt: 

In [ ]:
def get_model(num_keypoints, weights_path=None):
    
    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes = 2, # Background is the first class, object is the second class
                                                                   rpn_anchor_generator=anchor_generator)

    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)        
        
    return model

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

KEYPOINTS_FOLDER_TRAIN = '/Users/oliverbaltzer/Google Drive/Shared drives/Cooley_Lab/Hybrid Speckling_Spot-Vein-Project/JoshuaShin_PetalPhotos/'

# write a grab and split function 

(test, train) = shuffle_split_train_test(grab_all_data(KEYPOINTS_FOLDER_TRAIN), 0.3)

dataset_train = ClassDataset(train, transform=train_transform(), demo=False)
dataset_test = ClassDataset(test, transform=None, demo=False)

print(len(dataset_train))
print(len(dataset_test))
data_loader_train = DataLoader(dataset_train, batch_size=3, shuffle=True, collate_fn=collate_fn)
data_loader_test = DataLoader(dataset_test, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = get_model(num_keypoints = 4)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)
num_epochs = 50

metrics = []
errors = []

for epoch in range(num_epochs):
    metric = train_one_epoch(model, optimizer, data_loader_train, device, epoch, print_freq=1000)
    metrics.append(metric)
    lr_scheduler.step()
    error = evaluate(model, data_loader_test, device)
    errors.append(error)
    
# Save model weights after training


In [ ]:
keypoint_errors = [error.errors for error in errors]

keypoint_errors_by_keypoint = list(zip(*keypoint_errors))



# Plot errors for each keypoint
plt.figure(figsize=(10, 6))
print((keypoint_errors_by_keypoint))

plt.plot(keypoint_errors_by_keypoint[0], label=f'center_vein_bottom')
plt.plot(keypoint_errors_by_keypoint[1], label=f'center_vein_top')
plt.plot(keypoint_errors_by_keypoint[2], label=f'left_cut_edge')
plt.plot(keypoint_errors_by_keypoint[3], label=f'right_cut_edge')


plt.xlabel('Epoch')
plt.ylabel('Error (pixels)')
plt.title('Keypoint Errors Over Epochs')
plt.legend()
plt.grid()
plt.show()


In [ ]:
WEIGHTS_FILE = ""

torch.save(model.state_dict(), WEIGHTS_FILE)

In [ ]:
WEIGHTS_FILE = "keypointsrcnn_weights_33epochs.pth"

model = get_model(4,weights_path=WEIGHTS_FILE)


In [ ]:
iterator = iter(data_loader_test)
images, targets = next(iterator)
images = list(image.to(device) for image in images)

with torch.no_grad():
    model.to(device)
    model.eval()
    output = model(images)

# print("Predictions: \n", output)

In [ ]:
image = (images[0].permute(1,2,0).detach().cpu().numpy() * 255).astype(np.uint8)
scores = output[0]['scores'].detach().cpu().numpy()

high_scores_idxs = np.where(scores > 0.7)[0].tolist() # Indexes of boxes with scores > 0.7
if not high_scores_idxs:
    print("no viable keypoints")

post_nms_idxs = torchvision.ops.nms(output[0]['boxes'][high_scores_idxs], output[0]['scores'][high_scores_idxs], 0.3).cpu().numpy() # Indexes of boxes left after applying NMS (iou_threshold=0.3)

# Below, in output[0]['keypoints'][high_scores_idxs][post_nms_idxs] and output[0]['boxes'][high_scores_idxs][post_nms_idxs]
# Firstly, we choose only those objects, which have score above predefined threshold. This is done with choosing elements with [high_scores_idxs] indexes
# Secondly, we choose only those objects, which are left after NMS is applied. This is done with choosing elements with [post_nms_idxs] indexes

keypoints = []
for kps in output[0]['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    keypoints.append([list(map(int, kp[:2])) for kp in kps])


bboxes = []
for bbox in output[0]['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    bboxes.append(list(map(int, bbox.tolist())))
    
visualize_kp(image, bboxes, keypoints)